In [2]:
library(dplyr)
library(stringr)
library(openxlsx)

set.seed(123)

n <- 2500


In [3]:
# MASTER KANWIL ==========
kanwil <- tibble(
  KANWIL = c(
    "ACEH","SUMATERA UTARA","SUMATERA BARAT","RIAU",
    "JAMBI","SUMATERA SELATAN","LAMPUNG",
    "BANTEN","DKI JAKARTA","JAWA BARAT",
    "JAWA TENGAH","DI YOGYAKARTA","JAWA TIMUR",
    "BALI","NTB","NTT",
    "KALBAR","KALTENG","KALSEL","KALTIM","KALTARA",
    "SULUT","GORONTALO","SULTENG","SULSEL","SULTRA",
    "MALUKU","MALUKU UTARA","PAPUA","PAPUA BARAT"
  ),

  LAT = c(
    5.55,3.59,-0.95,0.51,-1.59,-2.99,-5.45,
    -6.12,-6.20,-6.91,-6.99,-7.80,-7.25,
    -8.65,-8.58,-10.17,
    -0.02,-2.21,-3.32,-0.50,3.30,
    1.47,0.54,-0.89,-5.14,-3.99,
    -3.69,0.79,-2.54,-0.86
  ),

  LON = c(
    95.32,98.67,100.35,101.45,103.61,104.75,105.27,
    106.15,106.81,107.61,110.42,110.37,112.75,
    115.22,116.10,123.60,
    109.34,113.92,114.59,117.15,117.63,
    124.84,123.06,119.87,119.41,122.51,
    128.18,127.39,140.71,134.08
  )
)


In [4]:
# MEMBUAT KANCAB ==========
kancab <- kanwil %>%
  rowwise() %>%
  do({

    jumlah <- sample(3:6,1)

    data.frame(
      KANWIL = .$KANWIL,
      LAT = .$LAT,
      LON = .$LON,
      KANCAB = paste0("KANCAB ",1:jumlah)
    )

  }) %>%
  ungroup()


In [5]:
# BOBOT AGAR JAWA LEBIH BANYAK ==========
bobot <- ifelse(
  kancab$KANWIL %in%
    c("BANTEN","DKI JAKARTA","JAWA BARAT",
      "JAWA TENGAH","DI YOGYAKARTA","JAWA TIMUR"),
  4,
  1
)


In [6]:
# DATA DUMMY ==========
dummy <- tibble(ID = 1:n)

dummy <- dummy %>%

  mutate(
    idx = sample(
      1:nrow(kancab),
      n,
      replace = TRUE,
      prob = bobot
    )
  ) %>%

  mutate(

    KANWIL = kancab$KANWIL[idx],
    KANCAB = paste(
      KANWIL,
      kancab$KANCAB[idx]
    ),

    lat0 = kancab$LAT[idx],
    lon0 = kancab$LON[idx]

  ) %>%

  mutate(

    `NAMA MITRA` =
      paste(
        sample(c(
          "PT",
          "CV"
        ),n,TRUE),

        sample(c(
          "MAKMUR",
          "SEJAHTERA",
          "BERKAH",
          "MANDIRI",
          "HASIL TANI",
          "NUSANTARA",
          "PANGAN",
          "SENTOSA"
        ),n,TRUE),

        sample(c(
          "ABADI",
          "JAYA",
          "INDO",
          "GROUP"
        ),n,TRUE),

        str_pad(ID,4,pad="0")
      ),

    `ID REKANAN` =
      paste0(
        "RK",
        str_pad(ID,6,pad="0")
      ),

    `ALAMAT MITRA` =
      paste(
        "Jl.",
        sample(c(
          "Merdeka",
          "Diponegoro",
          "Sudirman",
          "Ahmad Yani",
          "Veteran",
          "Pahlawan"
        ),n,TRUE),

        "No.",
        sample(1:200,n,TRUE)
      ),

    LATITUDE =
      lat0 + rnorm(n,0,0.15),

    LONGITUDE =
      lon0 + rnorm(n,0,0.15),

    `KOORDINAT LOKASI` =
      paste0(
        round(LATITUDE,6),
        ", ",
        round(LONGITUDE,6)
      ),

    `Kapasitas Pengeringan (ton/hari)` =
      sample(seq(10,120,5),n,TRUE),

    `Kapasitas Penggilingan (ton/hari)` =
      sample(seq(5,80,5),n,TRUE)

  ) %>%

  select(

    KANWIL,
    KANCAB,
    `NAMA MITRA`,
    `ID REKANAN`,
    `ALAMAT MITRA`,
    `KOORDINAT LOKASI`,
    `Kapasitas Pengeringan (ton/hari)`,
    `Kapasitas Penggilingan (ton/hari)`

  )

head(dummy)


# A tibble: 6 × 8
  KANWIL         KANCAB     `NAMA MITRA` `ID REKANAN` `ALAMAT MITRA` `KOORDINAT LOKASI` Kapasitas Pengeringa…¹
  <chr>          <chr>      <chr>        <chr>        <chr>          <chr>                               <dbl>
1 PAPUA          PAPUA KAN… PT PANGAN I… RK000001     Jl. Ahmad Yan… -2.788443, 141.13…                     95
2 SUMATERA UTARA SUMATERA … CV NUSANTAR… RK000002     Jl. Sudirman … 3.850035, 98.5535…                     35
3 JAMBI          JAMBI KAN… PT SENTOSA … RK000003     Jl. Veteran N… -1.2763, 103.6579…                     60
4 KALBAR         KALBAR KA… CV SENTOSA … RK000004     Jl. Diponegor… 0.139591, 109.472…                     50
5 JAWA TENGAH    JAWA TENG… CV HASIL TA… RK000005     Jl. Sudirman … -7.052013, 110.24…                     25
6 DKI JAKARTA    DKI JAKAR… PT HASIL TA… RK000006     Jl. Veteran N… -6.288357, 106.87…                     40
# ℹ abbreviated name: ¹​`Kapasitas Pengeringan (ton/hari)`
# ℹ 1 more variable: `Kapasitas Pen

In [7]:
# Simpan ke Excel ==========
write.xlsx(dummy,
           "data/Data Dummy Mitra Indonesia.xlsx",
           overwrite = TRUE)
